# 07 Posterior Expected Utility Targets

This notebook computes factual posterior expected utility targets after notebook 06 has estimated the structural model and saved the reusable estimation bundle.

The previous notebook-07 counterfactual responsibility target `tilde_r` is no longer the main object. The new target is

$$
EU_{itj}=E\left[u_{itj}\mid \text{watch ratio}_{itj},x_{itj},B_{it},c_i,\hat\theta,\hat\phi\right].
$$

For each observed structural interaction, notebook 06 already saved the factual posterior responsibility `r_hat` and factual threshold `tau_hat`. This notebook combines them with the calibrated structural utility moments \(\hat\mu_{ik}\) and \(\hat\sigma_{ik}\) to compute:

$$
EU_{itj}=\hat{r}_{itj}E\left[u\mid u>\hat\tau_{it}\right]+(1-\hat{r}_{itj})E\left[u\mid u\leq \hat\tau_{it}\right],
$$

and the standardized target for notebook 08:

$$
q^{\mathrm{target}}_{itj}=\frac{EU_{itj}-\hat\mu_{i,k_1(j)}}{\hat\sigma_{i,k_1(j)}}.
$$

Notebook 08 will estimate the personalized level-2 and level-3 utility parameters by fitting normalized \(q_\omega(i,j)\) to this observed \(q^{\mathrm{target}}_{itj}\).

## 1. Setup

Required files produced by notebook 06:

| File | Contents |
|---|---|
| `structural_estimates_theta_phi.npz` | estimated structural utility parameters and ids |
| `structural_interactions.parquet` | row-level interaction features used by EM |
| `structural_estimation_arrays.npz` | `sigmas_by_user`, ids, and reusable arrays |
| `structural_baseline_posteriors.parquet` | factual posterior `r_hat` and threshold `tau_hat` |

This notebook does not solve new counterfactual thresholds and does not recompute `r_hat`. It only transforms the factual posterior state information into expected-utility targets.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from scipy.special import erf

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)

PROJECT_ROOT = Path("/Users/haozhangao/Desktop/RecSys Research")
OUTPUT_DIR = PROJECT_ROOT / "python_code_new" / "outputs"

STRUCTURAL_PARAM_PATH = OUTPUT_DIR / "structural_estimates_theta_phi.npz"
STRUCTURAL_INTERACTIONS_PATH = OUTPUT_DIR / "structural_interactions.parquet"
STRUCTURAL_ARRAYS_PATH = OUTPUT_DIR / "structural_estimation_arrays.npz"
BASELINE_POSTERIOR_PATH = OUTPUT_DIR / "structural_baseline_posteriors.parquet"
BUNDLE_METADATA_PATH = OUTPUT_DIR / "structural_estimation_bundle_metadata.json"

GNN_DATA_PATH = PROJECT_ROOT / "KuaiRec 2.0" / "data" / "processed" / "gnn_data.pt"
OBSERVED_TARGET_OUTPUT_PATH = OUTPUT_DIR / "semi_synth_observed_targets.parquet"
GNN_TARGET_TABLE_PATH = OUTPUT_DIR / "gnn_eu_targets.parquet"
GNN_TARGET_TENSOR_PATH = OUTPUT_DIR / "gnn_eu_targets.pt"
DIAGNOSTIC_OUTPUT_PATH = OUTPUT_DIR / "posterior_expected_utility_diagnostics.json"

SIGMA_FLOOR = 1e-8
CDF_EPS = 1e-12

required_paths = [
    STRUCTURAL_PARAM_PATH,
    STRUCTURAL_INTERACTIONS_PATH,
    STRUCTURAL_ARRAYS_PATH,
    BASELINE_POSTERIOR_PATH,
    GNN_DATA_PATH,
]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Notebook 07 expects notebook 06's saved structural bundle. "
        "Rerun notebook 06 first. Missing files:\n" + "\n".join(missing)
    )

## 2. Load the Structural Bundle

The id maps and compact indices must agree across the parameter, array, interaction, and posterior files. The saved `row_id` remains the alignment key back to the GNN edge rows.

In [2]:
params = np.load(STRUCTURAL_PARAM_PATH)
arrays = np.load(STRUCTURAL_ARRAYS_PATH)
interaction_df = pd.read_parquet(STRUCTURAL_INTERACTIONS_PATH)
baseline_posterior = pd.read_parquet(BASELINE_POSTERIOR_PATH)
metadata = json.loads(BUNDLE_METADATA_PATH.read_text()) if BUNDLE_METADATA_PATH.exists() else {}

user_ids = params["user_ids"].astype(int)
cat_ids = params["cat_ids"].astype(int)
reference_cat_idx = int(params["reference_cat_idx"][0])

bundle_user_ids = arrays["user_ids"].astype(int)
bundle_cat_ids = arrays["cat_ids"].astype(int)
if not np.array_equal(user_ids, bundle_user_ids):
    raise ValueError("user_ids mismatch between structural params and estimation arrays")
if not np.array_equal(cat_ids, bundle_cat_ids):
    raise ValueError("cat_ids mismatch between structural params and estimation arrays")

sigmas_by_user = arrays["sigmas_by_user"].astype(np.float64)
N_users = len(user_ids)
K_cat = len(cat_ids)

if sigmas_by_user.shape != (N_users, K_cat):
    raise ValueError(f"sigmas_by_user shape {sigmas_by_user.shape}, expected {(N_users, K_cat)}")
if len(interaction_df) != len(baseline_posterior):
    raise ValueError("interaction_df and baseline_posterior have different row counts")

for col in ["user_idx_int", "cat_idx_int", "sess_idx"]:
    interaction_df[col] = interaction_df[col].astype(np.int64)
    baseline_posterior[col] = baseline_posterior[col].astype(np.int64)

required_posterior_cols = [
    "row_id", "user_id", "video_id", "burst_id", "session", "sess_idx",
    "i_cat_level1_id", "user_idx_int", "cat_idx_int", "r_hat", "tau_hat",
]
missing_cols = [col for col in required_posterior_cols if col not in baseline_posterior.columns]
if missing_cols:
    raise ValueError(f"Missing required baseline posterior columns: {missing_cols}")

if "row_id" not in interaction_df.columns:
    raise RuntimeError(
        "The saved notebook-06 bundle does not contain row_id. "
        "Rerun notebook 06 so targets can align exactly to gnn_data edge rows."
    )
if not np.array_equal(
    interaction_df["row_id"].to_numpy(dtype=np.int64),
    baseline_posterior["row_id"].to_numpy(dtype=np.int64),
):
    raise ValueError("interaction_df and baseline_posterior row_id order mismatch")
if baseline_posterior["row_id"].duplicated().any():
    raise ValueError("Duplicate row_id values in baseline posterior target table")

print("loaded bundle metadata:")
print(json.dumps(metadata, indent=2)[:2000])
print("interactions:", interaction_df.shape)
print("baseline posterior:", baseline_posterior.shape)
print("sigmas_by_user:", sigmas_by_user.shape)
print("N_users:", N_users)
print("K_cat:", K_cat)
print("reference category:", int(cat_ids[reference_cat_idx]), "at idx", reference_cat_idx)

loaded bundle metadata:
{
  "n_users": 1777,
  "n_categories": 39,
  "n_interactions": 4325560,
  "n_sessions": 81566,
  "burn_in_days": 2,
  "min_obs_per_user": 2000,
  "score_weight_spec": "naive",
  "score_weight_source": "manual::naive_(1,1,1,-1)",
  "score_weights": {
    "y_complete": 1.0,
    "y_long": 1.0,
    "y_rewatch": 1.0,
    "y_neg": -1.0
  },
  "baseline_observed_ll": -4325944.288365455,
  "row_id_alignment": "row_id equals the original processed_data row index and gnn_data edge row",
  "paths": {
    "interactions": "/Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/structural_interactions.parquet",
    "sessions": "/Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/structural_sessions.parquet",
    "arrays": "/Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/structural_estimation_arrays.npz",
    "baseline_posteriors": "/Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/structural_baseline_posteriors.parque

## 3. Reconstruct \(\hat\mu_{ik}\) and Row-Level Structural Quantities

Notebook 06 identifies utility location relative to the reference category. We reconstruct the same centered \(\hat\mu\) matrix, then gather the row-level \(\hat\mu_{i,k_1(j)}\), \(\hat\sigma_{i,k_1(j)}\), \(\hat\tau_{it}\), and \(\hat{r}_{itj}\).

In [3]:
theta = {
    "b_cat": params["theta_b_cat"].astype(np.float64),
    "U": params["theta_U"].astype(np.float64),
    "V": params["theta_V"].astype(np.float64),
    "reference_cat_idx": reference_cat_idx,
}


def theta_to_mu_matrix(theta, n_users=N_users, k_cat=K_cat, center_reference=True):
    mu = theta["b_cat"][None, :] + theta["U"] @ theta["V"].T
    if mu.shape != (n_users, k_cat):
        raise ValueError(f"mu shape {mu.shape}, expected {(n_users, k_cat)}")
    if center_reference:
        mu = mu - mu[:, [theta["reference_cat_idx"]]]
    return mu


mu_matrix = theta_to_mu_matrix(theta)

row_user_idx = baseline_posterior["user_idx_int"].to_numpy(dtype=np.int64, copy=False)
row_cat_idx = baseline_posterior["cat_idx_int"].to_numpy(dtype=np.int64, copy=False)
r_hat = baseline_posterior["r_hat"].to_numpy(dtype=np.float64, copy=False)
tau_hat = baseline_posterior["tau_hat"].to_numpy(dtype=np.float64, copy=False)

if row_user_idx.min() < 0 or row_user_idx.max() >= N_users:
    raise ValueError("user_idx_int outside structural user range")
if row_cat_idx.min() < 0 or row_cat_idx.max() >= K_cat:
    raise ValueError("cat_idx_int outside structural category range")

mu_hat_obs = mu_matrix[row_user_idx, row_cat_idx]
sigma_hat_obs = sigmas_by_user[row_user_idx, row_cat_idx]
sigma_hat_obs = np.maximum(sigma_hat_obs, SIGMA_FLOOR)

finite_mask = np.isfinite(mu_hat_obs) & np.isfinite(sigma_hat_obs) & np.isfinite(tau_hat) & np.isfinite(r_hat)
if not finite_mask.all():
    raise ValueError(f"Nonfinite row-level structural quantities: dropped would be {int((~finite_mask).sum())}")
if np.any((r_hat < -1e-8) | (r_hat > 1.0 + 1e-8)):
    raise ValueError("r_hat contains values outside [0, 1]")

r_hat = np.clip(r_hat, 0.0, 1.0)

print("mu_hat_obs summary:", pd.Series(mu_hat_obs).describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).to_dict())
print("sigma_hat_obs summary:", pd.Series(sigma_hat_obs).describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).to_dict())
print("tau_hat summary:", pd.Series(tau_hat).describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).to_dict())
print("r_hat summary:", pd.Series(r_hat).describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).to_dict())

mu_hat_obs summary: {'count': 4325560.0, 'mean': 2.071214602069675, 'std': 1.788628864580397, 'min': -11.540832393222566, '1%': -2.1389690134186017, '5%': -0.6829084548292224, '50%': 2.145718766819587, '95%': 4.556812635588015, '99%': 5.079334771692513, 'max': 9.52878880349742}


sigma_hat_obs summary: {'count': 4325560.0, 'mean': 1.0243094179969727, 'std': 0.18217195049670815, 'min': 0.053732776179309154, '1%': 0.5063010708360047, '5%': 0.7315371649048034, '50%': 1.0304527423274916, '95%': 1.2755346822563198, '99%': 1.4998713961003498, 'max': 3.810511776651529}
tau_hat summary: {'count': 4325560.0, 'mean': 1.7630977481182717, 'std': 1.1959427002765715, 'min': -7.805761814117432, '1%': -1.1041477918624878, '5%': -0.020973309874534607, '50%': 1.811284065246582, '95%': 3.356527328491211, '99%': 3.6730716228485107, 'max': 7.521847248077393}


r_hat summary: {'count': 4325560.0, 'mean': 0.6575610477815536, 'std': 0.38646142430882746, 'min': 0.0, '1%': 0.0, '5%': 8.263005317683046e-14, '50%': 0.8786556422710419, '95%': 0.9860413074493408, '99%': 0.9920122623443604, 'max': 1.0}


## 4. Compute Posterior Expected Utility

For each observed row, let

$$
a_{itj}=\frac{\hat\tau_{it}-\hat\mu_{i,k_1(j)}}{\hat\sigma_{i,k_1(j)}}.
$$

Under the Gaussian category-utility assumption,

$$
E\left[u\mid u>\tau\right]=\mu+\sigma\frac{\phi(a)}{1-\Phi(a)},
$$

and

$$
E\left[u\mid u\leq\tau\right]=\mu-\sigma\frac{\phi(a)}{\Phi(a)}.
$$

Then

$$
EU_{itj}=\hat{r}_{itj}E\left[u\mid u>\hat\tau_{it}\right]+(1-\hat{r}_{itj})E\left[u\mid u\leq\hat\tau_{it}\right],
$$

and

$$
q^{\mathrm{target}}_{itj}=\frac{EU_{itj}-\hat\mu_{i,k_1(j)}}{\hat\sigma_{i,k_1(j)}}.
$$

In [4]:
def standard_normal_pdf(x):
    x = np.asarray(x, dtype=np.float64)
    return (1.0 / np.sqrt(2.0 * np.pi)) * np.exp(-0.5 * x * x)


def standard_normal_cdf(x):
    x = np.asarray(x, dtype=np.float64)
    return 0.5 * (1.0 + erf(x / np.sqrt(2.0)))


a_obs = (tau_hat - mu_hat_obs) / sigma_hat_obs
cdf = standard_normal_cdf(a_obs)
survival = 1.0 - cdf
cdf_clipped = np.clip(cdf, CDF_EPS, 1.0)
survival_clipped = np.clip(survival, CDF_EPS, 1.0)
pdf_a = standard_normal_pdf(a_obs)

upper_mills = pdf_a / survival_clipped
lower_mills = pdf_a / cdf_clipped
E_upper = mu_hat_obs + sigma_hat_obs * upper_mills
E_lower = mu_hat_obs - sigma_hat_obs * lower_mills
EU = r_hat * E_upper + (1.0 - r_hat) * E_lower
q_target = (EU - mu_hat_obs) / sigma_hat_obs
posterior_confidence_weight = 2.0 * np.abs(r_hat - 0.5)

if not (np.isfinite(E_upper).all() and np.isfinite(E_lower).all() and np.isfinite(EU).all() and np.isfinite(q_target).all()):
    raise RuntimeError("Nonfinite posterior expected utility output")

clip_diag = {
    "cdf_below_eps": int((cdf < CDF_EPS).sum()),
    "survival_below_eps": int((survival < CDF_EPS).sum()),
    "cdf_eps": float(CDF_EPS),
}

print("clip diagnostics:", clip_diag)
print("E_upper summary:", pd.Series(E_upper).describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).to_dict())
print("E_lower summary:", pd.Series(E_lower).describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).to_dict())
print("EU summary:", pd.Series(EU).describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).to_dict())
print("q_target summary:", pd.Series(q_target).describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).to_dict())
print("corr(r_hat, EU):", float(np.corrcoef(r_hat, EU)[0, 1]))
print("corr(r_hat, q_target):", float(np.corrcoef(r_hat, q_target)[0, 1]))

clip diagnostics: {'cdf_below_eps': 3, 'survival_below_eps': 6787, 'cdf_eps': 1e-12}
E_upper summary: {'count': 4325560.0, 'mean': 2.7647941600150965, 'std': 1.449707912049429, 'min': -7.607520136145533, '1%': -0.5890192967739449, '5%': 0.60959086961449, '50%': 2.779465761857151, '95%': 4.833698880335947, '99%': 5.297727775466204, 'max': 10.459112200935511}


E_lower summary: {'count': 4325560.0, 'mean': 0.9879933339471955, 'std': 1.3844527732111602, 'min': -11.545487538961984, '1%': -2.4825252040759818, '5%': -1.1141717533779314, '50%': 1.0742626124373205, '95%': 2.802597722891746, '99%': 3.1051444106957495, 'max': 6.642277844271979}


EU summary: {'count': 4325560.0, 'mean': 2.167807153501078, 'std': 1.9262397036840386, 'min': -11.545487170656655, '1%': -2.2996836499943987, '5%': -0.8841725104045227, '50%': 2.3563921806218247, '95%': 4.763730269234737, '99%': 5.247886899805734, 'max': 10.328987763381434}
q_target summary: {'count': 4325560.0, 'mean': 0.09369276209911595, 'std': 0.46563957209330903, 'min': -3.389708625261914, '1%': -1.2463597220758322, '5%': -0.8831822001661, '50%': 0.2348497193130427, '95%': 0.6304297651835505, '99%': 0.7495593317147571, 'max': 0.9106165315274863}
corr(r_hat, EU): 0.7777004567075981


corr(r_hat, q_target): 0.8011346607396336


## 5. Save Observed Targets and GNN-Aligned Tensors

The observed target table feeds notebook 08's utility-specification fit. The GNN table/tensor files provide edge-aligned labels for scalable ML training. The tensor file stores both latent-scale `EU` and standardized `q_target`.

In [5]:
observed_target_df = baseline_posterior.copy()
observed_target_df["r"] = r_hat.astype(np.float32)
observed_target_df["mu_hat"] = mu_hat_obs.astype(np.float32)
observed_target_df["sigma_hat"] = sigma_hat_obs.astype(np.float32)
observed_target_df["a_obs"] = a_obs.astype(np.float32)
observed_target_df["E_upper"] = E_upper.astype(np.float32)
observed_target_df["E_lower"] = E_lower.astype(np.float32)
observed_target_df["EU"] = EU.astype(np.float32)
observed_target_df["q_target"] = q_target.astype(np.float32)
observed_target_df["posterior_confidence_weight"] = posterior_confidence_weight.astype(np.float32)
observed_target_df["target_name"] = "posterior_expected_utility"
observed_target_df.to_parquet(OBSERVED_TARGET_OUTPUT_PATH, index=False)

import torch

gnn_data_for_shape = torch.load(GNN_DATA_PATH, map_location="cpu", weights_only=False)
num_gnn_edges = int(gnn_data_for_shape["edge_index"].shape[1])
row_id = observed_target_df["row_id"].to_numpy(dtype=np.int64)
if row_id.min() < 0 or row_id.max() >= num_gnn_edges:
    raise ValueError(
        f"row_id is outside gnn_data edge range: min={row_id.min()}, max={row_id.max()}, num_edges={num_gnn_edges}"
    )
if len(np.unique(row_id)) != len(row_id):
    raise ValueError("Duplicate row_id values; cannot write aligned GNN target tensor")

target_eu = np.full(num_gnn_edges, np.nan, dtype=np.float32)
target_q = np.full(num_gnn_edges, np.nan, dtype=np.float32)
target_r_hat = np.full(num_gnn_edges, np.nan, dtype=np.float32)
target_weight = np.full(num_gnn_edges, np.nan, dtype=np.float32)
target_mask = np.zeros(num_gnn_edges, dtype=bool)

target_eu[row_id] = observed_target_df["EU"].to_numpy(dtype=np.float32)
target_q[row_id] = observed_target_df["q_target"].to_numpy(dtype=np.float32)
target_r_hat[row_id] = observed_target_df["r_hat"].to_numpy(dtype=np.float32)
target_weight[row_id] = observed_target_df["posterior_confidence_weight"].to_numpy(dtype=np.float32)
target_mask[row_id] = True

gnn_target_table = observed_target_df[[
    "row_id", "user_id", "video_id", "burst_id", "session", "sess_idx", "i_cat_level1_id",
    "r_hat", "tau_hat", "mu_hat", "sigma_hat", "a_obs", "EU", "q_target", "posterior_confidence_weight",
]].copy()
gnn_target_table["target_available"] = True
gnn_target_table.to_parquet(GNN_TARGET_TABLE_PATH, index=False)

torch.save({
    "y_EU": torch.from_numpy(target_eu),
    "y_q_target": torch.from_numpy(target_q),
    "y_r_hat": torch.from_numpy(target_r_hat),
    "target_weight_confidence": torch.from_numpy(target_weight),
    "target_mask": torch.from_numpy(target_mask),
    "target_edge_idx": torch.from_numpy(row_id),
    "target_name": "posterior_expected_utility",
    "standardized_target_name": "q_target",
    "alignment": "row_id equals gnn_data edge row index",
    "gnn_data_path": str(GNN_DATA_PATH),
    "observed_target_table_path": str(OBSERVED_TARGET_OUTPUT_PATH),
    "target_table_path": str(GNN_TARGET_TABLE_PATH),
    "n_targets": int(target_mask.sum()),
    "num_gnn_edges": int(num_gnn_edges),
}, GNN_TARGET_TENSOR_PATH)

print("saved observed EU targets:", OBSERVED_TARGET_OUTPUT_PATH)
print("saved GNN target table:", GNN_TARGET_TABLE_PATH)
print("saved GNN target tensor:", GNN_TARGET_TENSOR_PATH)

saved observed EU targets: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/semi_synth_observed_targets.parquet
saved GNN target table: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/gnn_eu_targets.parquet
saved GNN target tensor: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/gnn_eu_targets.pt


## 6. Diagnostics

These checks summarize the new target and preserve enough metadata for notebook 08 to verify that it is fitting the intended object.

In [6]:
def safe_corr(x, y):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        return np.nan
    return float(np.corrcoef(x[mask], y[mask])[0, 1])


diag = {
    "target": "posterior_expected_utility",
    "structural_param_path": str(STRUCTURAL_PARAM_PATH),
    "bundle_metadata_path": str(BUNDLE_METADATA_PATH),
    "n_users": int(N_users),
    "n_categories": int(K_cat),
    "n_interactions": int(len(observed_target_df)),
    "num_gnn_edges": int(num_gnn_edges),
    "n_gnn_targets": int(target_mask.sum()),
    "target_coverage_share": float(target_mask.mean()),
    "sigma_floor": float(SIGMA_FLOOR),
    "truncated_normal_clip": clip_diag,
    "r_hat_mean": float(np.mean(r_hat)),
    "r_hat_std": float(np.std(r_hat)),
    "EU_mean": float(np.mean(EU)),
    "EU_std": float(np.std(EU)),
    "EU_min": float(np.min(EU)),
    "EU_max": float(np.max(EU)),
    "q_target_mean": float(np.mean(q_target)),
    "q_target_std": float(np.std(q_target)),
    "q_target_min": float(np.min(q_target)),
    "q_target_max": float(np.max(q_target)),
    "corr_r_hat_EU": safe_corr(r_hat, EU),
    "corr_r_hat_q_target": safe_corr(r_hat, q_target),
    "corr_tau_hat_EU": safe_corr(tau_hat, EU),
    "observed_target_output_path": str(OBSERVED_TARGET_OUTPUT_PATH),
    "gnn_target_table_path": str(GNN_TARGET_TABLE_PATH),
    "gnn_target_tensor_path": str(GNN_TARGET_TENSOR_PATH),
}
DIAGNOSTIC_OUTPUT_PATH.write_text(json.dumps(diag, indent=2))

print("saved diagnostics:", DIAGNOSTIC_OUTPUT_PATH)
print(json.dumps(diag, indent=2))

saved diagnostics: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/posterior_expected_utility_diagnostics.json
{
  "target": "posterior_expected_utility",
  "structural_param_path": "/Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/structural_estimates_theta_phi.npz",
  "bundle_metadata_path": "/Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/structural_estimation_bundle_metadata.json",
  "n_users": 1777,
  "n_categories": 39,
  "n_interactions": 4325560,
  "num_gnn_edges": 12527912,
  "n_gnn_targets": 4325560,
  "target_coverage_share": 0.3452738173767504,
  "sigma_floor": 1e-08,
  "truncated_normal_clip": {
    "cdf_below_eps": 3,
    "survival_below_eps": 6787,
    "cdf_eps": 1e-12
  },
  "r_hat_mean": 0.6575610477815536,
  "r_hat_std": 0.3864613796367818,
  "EU_mean": 2.167807153501078,
  "EU_std": 1.9262394810261272,
  "EU_min": -11.545487170656655,
  "EU_max": 10.328987763381434,
  "q_target_mean": 0.09369276209911595,
  "q_targ

## 7. Quick Sanity Checks

In [7]:
summary_by_cat = (
    observed_target_df
    .groupby("i_cat_level1_id", observed=True)
    .agg(
        n=("user_id", "size"),
        r_hat_mean=("r_hat", "mean"),
        tau_hat_mean=("tau_hat", "mean"),
        mu_hat_mean=("mu_hat", "mean"),
        sigma_hat_mean=("sigma_hat", "mean"),
        EU_mean=("EU", "mean"),
        q_target_mean=("q_target", "mean"),
        q_target_std=("q_target", "std"),
    )
    .reset_index()
)
display(summary_by_cat.sort_values("n", ascending=False).head(15))

summary_by_user = (
    observed_target_df
    .groupby("user_id", observed=True)
    .agg(
        n=("user_id", "size"),
        r_hat_mean=("r_hat", "mean"),
        EU_mean=("EU", "mean"),
        q_target_mean=("q_target", "mean"),
        q_target_std=("q_target", "std"),
    )
    .reset_index()
)
display(summary_by_user[["n", "r_hat_mean", "EU_mean", "q_target_mean", "q_target_std"]].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))

observed_target_df[["r_hat", "tau_hat", "mu_hat", "sigma_hat", "a_obs", "E_upper", "E_lower", "EU", "q_target"]].head()

,i_cat_level1_id,n,r_hat_mean,tau_hat_mean,mu_hat_mean,sigma_hat_mean,EU_mean,q_target_mean,q_target_std
28,28,787833,0.714334,1.753598,2.415464,1.087317,2.456007,0.037120,0.471918
8,8,460007,0.720298,1.812502,2.474727,1.114420,2.544527,0.060635,0.460547
6,6,296658,0.676722,1.752032,2.066705,0.897971,2.129652,0.072699,0.462756
33,34,258844,0.681996,1.810587,2.277046,1.134365,2.388042,0.091812,0.462443
5,5,205299,0.680652,1.776823,2.174014,1.125857,2.297439,0.103724,0.472950
1,1,182761,0.687356,1.769272,2.104458,0.955886,2.193880,0.095514,0.463672
26,26,181260,0.680075,1.787986,2.087097,0.929878,2.173871,0.095970,0.464316
7,7,177889,0.685478,1.781960,2.160103,1.095680,2.282550,0.108420,0.466729
11,11,160784,0.664224,1.762683,2.043808,1.031955,2.155096,0.107044,0.467202
12,12,148618,0.610985,1.715194,1.826530,0.923817,1.934904,0.111183,0.456294


,n,r_hat_mean,EU_mean,q_target_mean,q_target_std
count,1777.000000,1777.000000,1777.000000,1777.000000,1777.000000
mean,2434.192459,0.654777,2.162362,0.093650,0.435813
std,514.185348,0.244762,1.688656,0.088511,0.142942
min,2000.000000,0.001449,-10.002311,-0.142558,0.116091
1%,2007.000000,0.053986,-2.000886,-0.097448,0.146392
5%,2025.800000,0.189400,-0.651081,-0.056966,0.188919
50%,2312.000000,0.704831,2.220812,0.100639,0.466436
95%,3251.000000,0.935018,4.499804,0.212646,0.630199
99%,4068.960000,0.951298,4.759327,0.228380,0.664140
max,13215.000000,0.963823,6.417454,0.250832,0.675871


,r_hat,tau_hat,mu_hat,sigma_hat,a_obs,E_upper,E_lower,EU,q_target
0,0.935666,2.073529,2.791307,1.011565,-0.709573,3.203574,1.478498,3.092592,0.297841
1,0.944520,2.073529,3.090969,1.088675,-0.934568,3.431140,1.487368,3.323300,0.213407
2,0.961293,2.073529,2.945219,0.981772,-0.887874,3.270165,1.535308,3.203014,0.262581
3,0.965687,2.073529,3.090969,1.088675,-0.934568,3.431140,1.487368,3.364444,0.251200
4,0.965552,2.073529,3.090969,1.088675,-0.934568,3.431140,1.487368,3.364181,0.250958
